In [38]:
import pandas as pd

#pip install scikit-learn

In [39]:
# Célula 2 — Carregamento do dataset

# Opção 1: CSV
df = pd.read_csv("br_seeg_emissoes_brasil.csv")

# Visualizar primeiras linhas
df.head()

,ano,nivel_1,nivel_2,nivel_3,nivel_4,nivel_5,nivel_6,tipo_emissao,gas,atividade_economica,produto,emissao
0,1970,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,230462.17
1,1971,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,226016.30
2,1972,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,220101.20
3,1973,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,214195.56
4,1974,Agropecuária,Cultivo do Arroz,Diretas,Outros,Vegetal,Arroz,Emissão,CH4 (t),NaN,NaN,186862.84


In [40]:
# Célula 3 — Seleção das colunas relevantes

df = df[["ano", "nivel_1", "gas", "emissao"]]

df.head()

,ano,nivel_1,gas,emissao
0,1970,Agropecuária,CH4 (t),230462.17
1,1971,Agropecuária,CH4 (t),226016.30
2,1972,Agropecuária,CH4 (t),220101.20
3,1973,Agropecuária,CH4 (t),214195.56
4,1974,Agropecuária,CH4 (t),186862.84


In [41]:
# Célula 4 — Filtrar apenas emissões de CO2

df_co2 = df[df["gas"] == "CO2 (t)"]

df_co2.head()
df_co2.count()


ano        37550
nivel_1    37550
gas        37550
emissao    30795
dtype: int64

In [42]:
# Célula 5 — Remoção de valores nulos

# df_co2 = df_co2.dropna()

# Verificar se ainda existem valores nulos
# df_co2.isnull().sum()
# df_co2.count()

# Achamos que não é necessário remover os valores nulos, pois eles representam anos ou atividades sem emissões de CO2, o que é relevante para a análise. Portanto, optamos por manter os valores nulos para preservar a integridade dos dados e evitar distorções na análise.

In [43]:
# Célula 6 — Aplicação de One-Hot Encoding

df_encoded = pd.get_dummies(
    # get_dummies é uma função do pandas que converte variáveis categóricas em variáveis dummy (0 ou 1)
    df_co2,
    columns=["nivel_1"],
    drop_first=True  # evita redundância
)

df_encoded.head()

,ano,gas,emissao,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos
14150,1970,CO2 (t),223538.51,False,False,False,False
14151,1971,CO2 (t),225473.06,False,False,False,False
14152,1972,CO2 (t),333445.33,False,False,False,False
14153,1973,CO2 (t),280380.69,False,False,False,False
14154,1974,CO2 (t),315281.04,False,False,False,False


In [44]:
# Célula 7 — Normalização da variável de resposta (Emissão)

df_encoded["Emissao_norm"] = (
    df_encoded["emissao"] - df_encoded["emissao"].min()
) / (
    df_encoded["emissao"].max()
    - df_encoded["emissao"].min()
)

df_encoded.head()

,ano,gas,emissao,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos,Emissao_norm
14150,1970,CO2 (t),223538.51,False,False,False,False,0.129892
14151,1971,CO2 (t),225473.06,False,False,False,False,0.129893
14152,1972,CO2 (t),333445.33,False,False,False,False,0.129938
14153,1973,CO2 (t),280380.69,False,False,False,False,0.129916
14154,1974,CO2 (t),315281.04,False,False,False,False,0.129931


In [45]:
# Célula 8 — Remover valores nulos (NaN)

df_encoded = df_encoded.dropna()

print(f"Dados após remover NaN: {len(df_encoded)} linhas")
df_encoded.head()

Dados após remover NaN: 30795 linhas


,ano,gas,emissao,nivel_1_Energia,nivel_1_Mudança de Uso da Terra e Floresta,nivel_1_Processos Industriais,nivel_1_Resíduos,Emissao_norm
14150,1970,CO2 (t),223538.51,False,False,False,False,0.129892
14151,1971,CO2 (t),225473.06,False,False,False,False,0.129893
14152,1972,CO2 (t),333445.33,False,False,False,False,0.129938
14153,1973,CO2 (t),280380.69,False,False,False,False,0.129916
14154,1974,CO2 (t),315281.04,False,False,False,False,0.129931


# Modelos de Regressão.

Regressão Linear Multipla.  Multiple Linear Regression (MLR)

In [46]:
X = df_encoded.drop(['emissao', 'gas'], axis=1)  # variáveis independentes (removendo a coluna alvo e 'gas' que é redundante)
y = df_encoded['emissao']               # variável alvo

In [47]:
# Célula 9 — Divisão dos dados

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
        # test_size=0.2 significa que 20% dos dados serão usados para teste e 80% para treino.
        # random_state=42 é uma semente para garantir que a divisão seja a mesma toda vez que o código for executado, permitindo reprodutibilidade.
)

In [48]:
# Célula 10 — Treinamento da Regressão Linear Múltipla

from sklearn.linear_model import LinearRegression

modelo_linear = LinearRegression()
modelo_linear.fit(X_train, y_train)



,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [49]:
# Célula 11 — Treinamento da Regressão Árvore de Decisão
from sklearn.tree import DecisionTreeRegressor 

modelo_arvore = DecisionTreeRegressor(max_depth=5)
modelo_arvore.fit(X_train, y_train)

# ANOTAÇOES:
# * A arevore de decisão com certeza é melhor que a arevore de regração linar 
# * pode ser que deva ser aplicado overfitting (overfitting é um modelo que decora os dados de treinamento) na arvore de decição, devemos avaliar a arvore antes de implementar ele pois quando chega um dado novo ele pode dar problemas de desempenho.

# se quiser visualizar a arvore de decisão é so aplicar

# from sklearn.tree import plot_tree
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12,8))
# plot_tree(modelo, filled=True)
# plt.show()
#



,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_le

In [50]:
# Célula 12 — Treinamento da Regressão Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
modelo_floresta = RandomForestRegressor(max_depth=5)        # controla overfitting 

modelo_floresta.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [51]:
# Célula 13 — Treinamento da Regressão XGBoost
import xgboost as xgb 
from sklearn.metrics import mean_squared_error

modelo_xgb = xgb.XGBRegressor(max_depth=5)  # max_depth controla overfitting, n_estimators é o número de árvores, learning_rate é a taxa de aprendizado
modelo_xgb.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [52]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Previsões
y_pred_linear = modelo_linear.predict(X_test)
y_pred_arvore = modelo_arvore.predict(X_test)
y_pred_floresta = modelo_floresta.predict(X_test)
y_pred_xgb = modelo_xgb.predict(X_test)

print("===== TAXA DE ERRO (RMSE) =====\n")

print("Regressão Linear:", np.sqrt(mean_squared_error(y_test, y_pred_linear)))
print("Árvore de Decisão:", np.sqrt(mean_squared_error(y_test, y_pred_arvore)))
print("Floresta:", np.sqrt(mean_squared_error(y_test, y_pred_floresta)))
print("XGBoost:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))

===== TAXA DE ERRO (RMSE) =====

Regressão Linear: 2.6168697274892995e-07
Árvore de Decisão: 3860403.255519237
Floresta: 3191874.918977059
XGBoost: 27240323.195821017
